# Day 3 · Deliver & Govern — Lab 1
# Agent Maturity & Delivery Lifecycle · Risk in Agentic Systems

Standard-library only, fully runnable top to bottom in VS Code / Jupyter.

**Contents**
1. Agent Maturity Model — a 5-level scoring rubric + calculator you can run against a real project
2. Delivery Lifecycle — stage-gate tracker (Discover → Pilot → Scale → Operate) with exit criteria
3. Risk Register for Agentic Systems — structured risk log with likelihood x impact scoring
4. Risk Heatmap (text-based) — visualize the register without extra plotting libraries
5. Failure-mode checklist — agent-specific risk taxonomy (hallucination, tool misuse, drift, etc.)


In [1]:
import json, time
from dataclasses import dataclass, field, asdict
from typing import Dict, Any, List, Optional
from datetime import date

print("Ready.")


Ready.


## 1. Agent Maturity Model

A simple 5-level maturity model, one dimension at a time. Score 0-4 per dimension,
average them for an overall maturity level. This mirrors how most agentic-AI maturity
frameworks are structured (ad hoc -> managed -> defined -> measured -> optimizing).

In [2]:
MATURITY_LEVELS = {
    0: "Ad hoc — agents built as one-off scripts, no monitoring, no owner",
    1: "Managed — basic logging, manual testing, single owner per agent",
    2: "Defined — standard build pattern, CI tests, documented handoff process",
    3: "Measured — SLOs tracked, automated eval suites, cost/latency dashboards",
    4: "Optimizing — continuous eval, automated rollback, portfolio-level reuse",
}

DIMENSIONS = [
    "strategy_alignment",
    "build_standards",
    "testing_evaluation",
    "observability",
    "governance_risk",
    "operations_scale",
]


@dataclass
class MaturityAssessment:
    project_name: str
    scores: Dict[str, int] = field(default_factory=dict)   # dimension -> 0..4

    def set_score(self, dimension: str, score: int):
        assert dimension in DIMENSIONS, f"unknown dimension: {dimension}"
        assert 0 <= score <= 4, "score must be 0-4"
        self.scores[dimension] = score

    def overall_level(self) -> float:
        if not self.scores:
            return 0.0
        return round(sum(self.scores.values()) / len(self.scores), 2)

    def report(self) -> str:
        lines = [f"Maturity Assessment: {self.project_name}", "-" * 50]
        for dim in DIMENSIONS:
            score = self.scores.get(dim, 0)
            level_label = MATURITY_LEVELS[score].split("—")[0].strip()
            lines.append(f"  {dim:22s}: level {score}  ({level_label})")
        overall = self.overall_level()
        lines.append("-" * 50)
        lines.append(f"  OVERALL MATURITY: {overall} / 4")
        return "\n".join(lines)


assessment = MaturityAssessment("Remittance Matching Agent")
assessment.set_score("strategy_alignment", 3)
assessment.set_score("build_standards", 2)
assessment.set_score("testing_evaluation", 2)
assessment.set_score("observability", 3)
assessment.set_score("governance_risk", 1)
assessment.set_score("operations_scale", 1)

print(assessment.report())


Maturity Assessment: Remittance Matching Agent
--------------------------------------------------
  strategy_alignment    : level 3  (Measured)
  build_standards       : level 2  (Defined)
  testing_evaluation    : level 2  (Defined)
  observability         : level 3  (Measured)
  governance_risk       : level 1  (Managed)
  operations_scale      : level 1  (Managed)
--------------------------------------------------
  OVERALL MATURITY: 2.0 / 4


## 2. Delivery Lifecycle — stage-gate tracker

A typical agentic-delivery lifecycle: **Discover -> Pilot -> Scale -> Operate**, each with
exit criteria that must be satisfied before advancing. This is a lightweight gate-checker
you can adapt to your own program's criteria.

In [3]:
@dataclass
class Gate:
    name: str
    criteria: List[str]

STAGE_GATES = [
    Gate("Discover", [
        "Business case documented",
        "Use case scoped with success metrics",
        "Data availability confirmed",
    ]),
    Gate("Pilot", [
        "Working prototype with real data (non-prod)",
        "Golden-set eval suite passing >= 80%",
        "Human-in-the-loop review process defined",
    ]),
    Gate("Scale", [
        "Production deployment with observability",
        "Risk register reviewed and signed off",
        "Rollback plan tested",
    ]),
    Gate("Operate", [
        "SLOs defined and monitored",
        "Ownership + on-call assigned",
        "Quarterly reuse/retirement review scheduled",
    ]),
]


@dataclass
class ProjectStatus:
    project_name: str
    completed_criteria: set = field(default_factory=set)   # set of criteria strings marked done

    def mark_done(self, criterion: str):
        self.completed_criteria.add(criterion)

    def current_stage(self) -> str:
        """Returns the furthest stage whose gate criteria are all satisfied."""
        furthest = "Not Started"
        for gate in STAGE_GATES:
            if all(c in self.completed_criteria for c in gate.criteria):
                furthest = gate.name
            else:
                break
        return furthest

    def next_gate_gaps(self) -> Optional[List[str]]:
        for gate in STAGE_GATES:
            missing = [c for c in gate.criteria if c not in self.completed_criteria]
            if missing:
                return {"gate": gate.name, "missing": missing}
        return None

    def report(self) -> str:
        lines = [f"Delivery Lifecycle: {self.project_name}", "-" * 50]
        lines.append(f"Current stage: {self.current_stage()}")
        gap = self.next_gate_gaps()
        if gap:
            gate_name = gap["gate"]
            lines.append(f"Next gate '{gate_name}' is blocked on:")
            for m in gap["missing"]:
                lines.append(f"    [ ] {m}")
        else:
            lines.append("All defined gates fully satisfied.")
        return "\n".join(lines)


proj = ProjectStatus("Remittance Matching Agent")
for c in ["Business case documented", "Use case scoped with success metrics",
          "Data availability confirmed", "Working prototype with real data (non-prod)",
          "Golden-set eval suite passing >= 80%"]:
    proj.mark_done(c)

print(proj.report())


Delivery Lifecycle: Remittance Matching Agent
--------------------------------------------------
Current stage: Discover
Next gate 'Pilot' is blocked on:
    [ ] Human-in-the-loop review process defined


---
## 3. Risk Register for Agentic Systems

A structured risk log: each risk has a **likelihood** (1-5), **impact** (1-5), category,
mitigation, and owner. Risk score = likelihood x impact. This is the standard shape used
in RAID logs and enterprise risk registers, adapted for agent-specific risk categories.

In [4]:
RISK_CATEGORIES = [
    "hallucination_accuracy",
    "data_privacy",
    "tool_misuse",
    "model_drift",
    "cost_overrun",
    "vendor_dependency",
    "human_oversight_gap",
    "regulatory_compliance",
]


@dataclass
class Risk:
    risk_id: str
    description: str
    category: str
    likelihood: int   # 1 (rare) - 5 (almost certain)
    impact: int        # 1 (negligible) - 5 (severe)
    mitigation: str
    owner: str
    status: str = "open"   # open | mitigating | closed

    @property
    def score(self) -> int:
        return self.likelihood * self.impact

    @property
    def severity(self) -> str:
        if self.score >= 15:
            return "CRITICAL"
        if self.score >= 9:
            return "HIGH"
        if self.score >= 4:
            return "MEDIUM"
        return "LOW"


class RiskRegister:
    def __init__(self):
        self.risks: List[Risk] = []

    def add(self, risk: Risk):
        self.risks.append(risk)

    def sorted_by_score(self) -> List[Risk]:
        return sorted(self.risks, key=lambda r: r.score, reverse=True)

    def report(self) -> str:
        lines = [f"{'ID':6s} {'SEVERITY':9s} {'SCORE':6s} {'CATEGORY':24s} {'STATUS':10s} DESCRIPTION"]
        for r in self.sorted_by_score():
            lines.append(f"{r.risk_id:6s} {r.severity:9s} {r.score:<6d} {r.category:24s} {r.status:10s} {r.description}")
        return "\n".join(lines)


register = RiskRegister()
register.add(Risk("R-01", "Agent hallucinates a matched invoice that does not exist",
                   "hallucination_accuracy", likelihood=3, impact=5,
                   mitigation="Require confidence>=0.95 + human review for AI-suggested matches",
                   owner="AI Lead"))
register.add(Risk("R-02", "Remittance email content sent to external LLM API without redaction",
                   "data_privacy", likelihood=2, impact=5,
                   mitigation="PII/PCI redaction pre-processing before any external call",
                   owner="Data Governance"))
register.add(Risk("R-03", "Agent granted write access misuses ERP posting tool",
                   "tool_misuse", likelihood=2, impact=4,
                   mitigation="Scoped tool permissions + dry-run mode + approval gate for posting",
                   owner="Platform Eng"))
register.add(Risk("R-04", "Matching accuracy degrades silently as remittance formats change",
                   "model_drift", likelihood=4, impact=3,
                   mitigation="Weekly golden-set regression run + alerting on accuracy drop",
                   owner="MLOps"))
register.add(Risk("R-05", "Unbounded retries on flaky ERP API spike LLM/API costs",
                   "cost_overrun", likelihood=3, impact=2,
                   mitigation="Circuit breaker + budget alert + max-retry caps",
                   owner="Platform Eng"))
register.add(Risk("R-06", "Single LLM vendor outage halts all agent operations",
                   "vendor_dependency", likelihood=2, impact=3,
                   mitigation="Fallback provider + graceful degradation to rules-only mode",
                   owner="Platform Eng"))

print(register.report())


ID     SEVERITY  SCORE  CATEGORY                 STATUS     DESCRIPTION
R-01   CRITICAL  15     hallucination_accuracy   open       Agent hallucinates a matched invoice that does not exist
R-04   HIGH      12     model_drift              open       Matching accuracy degrades silently as remittance formats change
R-02   HIGH      10     data_privacy             open       Remittance email content sent to external LLM API without redaction
R-03   MEDIUM    8      tool_misuse              open       Agent granted write access misuses ERP posting tool
R-05   MEDIUM    6      cost_overrun             open       Unbounded retries on flaky ERP API spike LLM/API costs
R-06   MEDIUM    6      vendor_dependency        open       Single LLM vendor outage halts all agent operations


## 4. Risk Heatmap (text-based)

A quick likelihood x impact grid, rendered as text so it works with zero plotting
dependencies — drop-in replacement is a real heatmap (matplotlib/seaborn) if you have it installed.

In [5]:
def render_heatmap(register: RiskRegister) -> str:
    grid = {(l, i): [] for l in range(1, 6) for i in range(1, 6)}
    for r in register.risks:
        grid[(r.likelihood, r.impact)].append(r.risk_id)

    lines = ["IMPACT ->      1        2        3        4        5"]
    for likelihood in range(5, 0, -1):
        row = [f"L={likelihood}  "]
        for impact in range(1, 6):
            ids = grid[(likelihood, impact)]
            cell = ",".join(ids) if ids else "."
            row.append(f"{cell:8s}")
        lines.append(" ".join(row))
    return "\n".join(lines)

print(render_heatmap(register))
print()
print("Legend: cell = risk IDs at that (Likelihood, Impact) coordinate. '.' = empty cell.")


IMPACT ->      1        2        3        4        5
L=5   .        .        .        .        .       
L=4   .        .        R-04     .        .       
L=3   .        R-05     .        .        R-01    
L=2   .        .        R-06     R-03     R-02    
L=1   .        .        .        .        .       

Legend: cell = risk IDs at that (Likelihood, Impact) coordinate. '.' = empty cell.


## 5. Failure-mode checklist — agent-specific risk taxonomy

A reusable checklist to run against any new agent before it goes to Pilot or Scale.
Each item maps to a `RISK_CATEGORIES` entry so gaps can be logged directly into the register above.

In [6]:
FAILURE_MODE_CHECKLIST = {
    "hallucination_accuracy": [
        "Does the agent cite/ground every factual claim in retrieved or tool-sourced data?",
        "Is there a confidence threshold below which the agent defers to a human?",
    ],
    "data_privacy": [
        "Is PII/PCI redacted or tokenized before leaving the trust boundary?",
        "Are prompts/responses logged with retention limits and access controls?",
    ],
    "tool_misuse": [
        "Are write/posting actions gated behind explicit approval or dry-run mode?",
        "Are tool permissions scoped to least privilege per agent role?",
    ],
    "model_drift": [
        "Is there a scheduled regression eval against a golden set?",
        "Is accuracy/latency tracked over time with alerting on regression?",
    ],
    "cost_overrun": [
        "Are there per-request and daily budget caps?",
        "Is there a circuit breaker for retry storms?",
    ],
    "vendor_dependency": [
        "Is there a fallback path if the primary model/provider is unavailable?",
    ],
    "human_oversight_gap": [
        "Is there a clearly defined human-in-the-loop step for high-impact decisions?",
    ],
    "regulatory_compliance": [
        "Has legal/compliance reviewed the use case against relevant regulations?",
    ],
}

def run_checklist(answers: Dict[str, Dict[str, bool]]) -> str:
    """answers: {category: {question: True/False}}"""
    lines = ["FAILURE-MODE CHECKLIST RESULTS", "-" * 50]
    total, passed = 0, 0
    for category, questions in FAILURE_MODE_CHECKLIST.items():
        lines.append(f"\n[{category}]")
        for q in questions:
            total += 1
            ok = answers.get(category, {}).get(q, False)
            passed += 1 if ok else 0
            mark = "PASS" if ok else "GAP "
            lines.append(f"  [{mark}] {q}")
    lines.append("\n" + "-" * 50)
    lines.append(f"Score: {passed}/{total} ({round(100*passed/total)}%)")
    return "\n".join(lines)


sample_answers = {
    "hallucination_accuracy": {
        "Does the agent cite/ground every factual claim in retrieved or tool-sourced data?": True,
        "Is there a confidence threshold below which the agent defers to a human?": True,
    },
    "data_privacy": {
        "Is PII/PCI redacted or tokenized before leaving the trust boundary?": False,
        "Are prompts/responses logged with retention limits and access controls?": True,
    },
    "tool_misuse": {
        "Are write/posting actions gated behind explicit approval or dry-run mode?": True,
        "Are tool permissions scoped to least privilege per agent role?": False,
    },
}

print(run_checklist(sample_answers))


FAILURE-MODE CHECKLIST RESULTS
--------------------------------------------------

[hallucination_accuracy]
  [PASS] Does the agent cite/ground every factual claim in retrieved or tool-sourced data?
  [PASS] Is there a confidence threshold below which the agent defers to a human?

[data_privacy]
  [GAP ] Is PII/PCI redacted or tokenized before leaving the trust boundary?
  [PASS] Are prompts/responses logged with retention limits and access controls?

[tool_misuse]
  [PASS] Are write/posting actions gated behind explicit approval or dry-run mode?
  [GAP ] Are tool permissions scoped to least privilege per agent role?

[model_drift]
  [GAP ] Is there a scheduled regression eval against a golden set?
  [GAP ] Is accuracy/latency tracked over time with alerting on regression?

[cost_overrun]
  [GAP ] Are there per-request and daily budget caps?
  [GAP ] Is there a circuit breaker for retry storms?

[vendor_dependency]
  [GAP ] Is there a fallback path if the primary model/provider is unav

---
### Exercises
1. Add a `target_score` per dimension to `MaturityAssessment` and report the gap to target.
2. Extend `ProjectStatus` to log a timestamped history of when each criterion was completed.
3. Add a `link_to_gate` field on `Risk` so unresolved CRITICAL/HIGH risks automatically block
   the corresponding stage gate in `ProjectStatus.next_gate_gaps()`.
